# Train the 3D CNN on a free Google Colab GPU (version 9)

**Before you start (on your laptop):**
1. Run the project up to `04_make_patches.py`. That creates `outputs/patches.npz` (version 9: every cube together with its mirror cube from the other side of the brain, 32 mm wide in every patient, plus 'not SOZ' cubes from the deep structures) and `03_train_baseline.py` creates `outputs/baseline_predictions.csv` (used for the comparison). A `patches.npz` of an older version is refused: run `01_label_contacts.py` ... `04_make_patches.py` again first.
2. Upload the whole `ez_localization_starter` folder to **Google Drive** -> `My Drive` (drive.google.com -> New -> Folder upload). It must contain `outputs/patches.npz`. You do **not** need to upload the MRI data, `synthetic_data` or `outputs_synthetic`. If an old copy of the folder is in Drive, delete it first.

**Settings in Colab (once):** menu **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save**.

Then run the cells below one by one (Shift + Enter). If Colab disconnects, run all cells again: training continues where it stopped (`--resume`).

### 1. Check that a GPU is connected

In [ ]:
!nvidia-smi
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"

### 2. Connect Google Drive (a window asks for permission - allow it)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Go to the project folder (change the path if you put it somewhere else in Drive)

In [ ]:
PROJECT = "/content/drive/MyDrive/ez_localization_starter"
import os
os.chdir(PROJECT)
print(os.getcwd())
print("patches.npz found:", os.path.exists("outputs/patches.npz"))

### 4. Install the one library Colab may not have

In [ ]:
!pip -q install nibabel

### 5. Train
Leave-one-patient-out: for every patient, 3 models are trained on the other patients and tested on him; then the final models are trained on all patients (`--final`, used by `08_predict_mri.py` for new MRIs). About 15-40 minutes on a T4 GPU. The first line must say `patients: 10`. Results are written to `outputs/` in your Drive.

In [ ]:
!EZ_DATA_SOURCE=local EZ_DATA_ROOT=. python 05_train_cnn.py --lopo --resume --final --epochs 40 --seeds 3

### 6. Look at the results

In [ ]:
import pandas as pd
pd.read_csv("outputs/cnn_metrics.csv")

### 7. Pack the results into one file
This makes `outputs/colab_results.zip` in your Drive (the result tables, the trained models and the log).

In [ ]:
import glob, os, zipfile
files = sorted(f for f in glob.glob("outputs/cnn_*") if os.path.isfile(f))
files += [f for f in ["outputs/logs/05_train_cnn.txt"] if os.path.exists(f)]
with zipfile.ZipFile("outputs/colab_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(len(files), "files ->", "outputs/colab_results.zip", round(os.path.getsize("outputs/colab_results.zip") / 1e6, 1), "MB")
print("models:", sum(f.endswith(".pt") for f in files), "  model card:", "outputs/cnn_model.json" in files)

**After training:** in Google Drive, download `ez_localization_starter/outputs/colab_results.zip`. On the laptop, extract it **into the project folder** (`ez_localization_starter`): it puts the files into `outputs\` (replace the old ones when Windows asks). Then run `python tools\model_report.py` and `python 08_predict_mri.py` there (README section 5).